## Pod affinity and anti-affinity — placement relative to *other pods*

Node affinity matches *node* labels. **Pod affinity** matches *other pods'* labels — "schedule me near (or away from) pods like this."

```yaml
affinity:
  podAntiAffinity:
    requiredDuringSchedulingIgnoredDuringExecution:
      - labelSelector: { matchLabels: { app: web } }
        topologyKey: kubernetes.io/hostname
```

Read it: "don't schedule me onto a node that already has an `app=web` pod." For a Deployment of `replicas: 3` with this rule, the result is **one pod per node** — the canonical replica spread.

**The `topologyKey` is the whole game** — it names a *node label* defining a "domain":

- **`kubernetes.io/hostname`** — each node its own domain → spread across hosts.
- **`topology.kubernetes.io/zone`** — each AZ → spread across zones.
- **`.../region`** — multi-region spreading.

`podAffinity` = same domain as a matching pod; `podAntiAffinity` = a *different* domain.

Both come in `required` and `preferred`. **Prefer `preferred` for spreading** — `required` anti-affinity with `hostname` and three replicas on a two-node cluster leaves one pod `Pending` forever.

Real uses: **spread a Deployment's replicas** (anti-affinity + hostname + own labels), **co-locate cache with app** (affinity), **keep two DBs off one node** (anti-affinity). Topology spread constraints (next section) often do the spreading job more concisely. On our map this is still the **affinity** chip in the Scheduling box — but scored against Pods rather than nodes.